# Notebook 4 : 04_Silver_to_Gold

##### Objective

###### This notebook transforms the cleaned Silver layer data into business-ready Gold tables.
###### The Gold layer contains aggregated data that supports business reporting, dashboards, and decision-making.

## Step 1 : Import Libraries

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

## Step 2 : Configuration

In [0]:
gold_path = "/Volumes/dbacademy/default/myvolume/gold"

silver_table = "silver_upi_transactions"

## Step 3 : Read Silver Table

In [0]:
silver_df = spark.table(silver_table)

display(silver_df.limit(5))

transaction_id,transaction_timestamp,sender_bank,receiver_bank,sender_upi,receiver_upi,amount,transaction_type,transaction_status,city,state,device_id,response_time_ms,failure_reason,load_timestamp,source_file
29fb4997-5692-4449-a7a3-5ac5a89017e2,2026-07-01T06:51:26.000Z,ICICI,SBI,an****30@icici,ka****22@oksbi,45291.0,P2P,SUCCESS,PUNE,MAHARASHTRA,DEV100030,2613,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
8965c981-9082-4302-80ea-a6143fb8ec8f,2026-07-03T20:42:48.000Z,SBI,AXIS,ro****66@oksbi,ri****23@axis,13260.0,BILL PAYMENT,SUCCESS,BANGALORE,KARNATAKA,DEV100057,441,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
738577eb-907d-411d-a632-b0fd92d700b9,2026-06-27T08:36:52.000Z,ICICI,AXIS,ma****95@icici,di****06@axis,30587.0,RECHARGE,SUCCESS,MUMBAI,MAHARASHTRA,DEV100041,128,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
3770dbc7-ec78-400b-8406-32ed60678044,2026-07-02T17:07:14.000Z,ICICI,AXIS,pr****73@icici,pr****41@axis,25455.0,MERCHANT,SUCCESS,BANGALORE,KARNATAKA,DEV100032,541,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
b1ebfc4b-f6d8-48b9-8a6a-6d5da03bb0ca,2026-07-06T03:02:40.000Z,ICICI,SBI,am****44@icici,di****22@oksbi,33119.0,P2P,FAILED,INDORE,MADHYA PRADESH,DEV100098,2648,Bank Server Down,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv


In [0]:
print("Number of rows are: ", silver_df.count())
print("Number of columns are: ", len(silver_df.columns))

Number of rows are:  19075
Number of columns are:  16


In [0]:
print(silver_df.columns)

['transaction_id', 'transaction_timestamp', 'sender_bank', 'receiver_bank', 'sender_upi', 'receiver_upi', 'amount', 'transaction_type', 'transaction_status', 'city', 'state', 'device_id', 'response_time_ms', 'failure_reason', 'load_timestamp', 'source_file']


In [0]:
silver_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- transaction_timestamp: timestamp (nullable = true)
 |-- sender_bank: string (nullable = true)
 |-- receiver_bank: string (nullable = true)
 |-- sender_upi: string (nullable = true)
 |-- receiver_upi: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- transaction_status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- response_time_ms: integer (nullable = true)
 |-- failure_reason: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



## Step 4 : Daily Transaction Summary

### 4.1 Extract Transaction Date

In [0]:
daily_df = silver_df.withColumn("transaction_date", to_date(col("transaction_timestamp")))

### 4.2 Create Daily Summary

In [0]:
daily_summary = (daily_df.groupBy("transaction_date")
                .agg(count("*").alias("total_transactions"),
                     
                sum("amount").alias("total_amount"),

                sum(when(col("transaction_status")=="SUCCESS",1)
                    .otherwise(0)).alias("successful_transactions"),

                sum(when(col("transaction_status")=="FAILED",1)
                    .otherwise(0)).alias("failed_transactions"),
                
                round(avg("amount"),2).alias("average_transaction_amount"),

                max("amount").alias("maximum_transaction_amount"),

                min("amount").alias("minimum_transaction_amount")
                )
)

### 4.3 Success Rate

In [0]:
daily_summary = daily_summary.withColumn(
    "success_rate",round((col("successful_transactions")*100)/
                          col("total_transactions"),2))

### 4.4 Failure Rate

In [0]:
daily_summary = daily_summary.withColumn(
    "failure_rate",round((col("failed_transactions")*100)/
                          col("total_transactions"),2))

### 4.5 : Verify

In [0]:
#sort data by date
daily_summary = daily_summary.orderBy("transaction_date")

display(daily_summary)

transaction_date,total_transactions,total_amount,successful_transactions,failed_transactions,average_transaction_amount,maximum_transaction_amount,minimum_transaction_amount,success_rate,failure_rate
2026-06-12,163,4101918.0,157,6,25165.14,49848.0,1094.0,96.32,3.68
2026-06-13,622,1.5424446E7,608,14,24798.14,49963.0,153.0,97.75,2.25
2026-06-14,644,1.6887342E7,623,21,26222.58,49708.0,112.0,96.74,3.26
2026-06-15,595,1.4886322E7,577,18,25019.03,49973.0,201.0,96.97,3.03
2026-06-16,685,1.7581704E7,668,17,25666.72,49963.0,151.0,97.52,2.48
2026-06-17,625,1.5846873E7,605,20,25355.0,49900.0,10.0,96.8,3.2
2026-06-18,684,1.7373323E7,654,30,25399.6,49890.0,28.0,95.61,4.39
2026-06-19,637,1.666387E7,618,19,26159.92,49920.0,48.0,97.02,2.98
2026-06-20,637,1.6351033E7,616,21,25668.81,49944.0,32.0,96.7,3.3
2026-06-21,594,1.481295E7,577,17,24937.63,49953.0,171.0,97.14,2.86


## Step 5 : Bank-wise Transaction Summary

### 5.1 – Create Bank Summary

In [0]:
bank_summary = (silver_df.groupBy("sender_bank")
    .agg(
        count("*").alias("total_transactions"),

        sum("amount").alias("total_amount"),

        round(avg("amount"),2).alias("average_transaction_amount"),

        max("amount").alias("maximum_transaction_amount"),

        min("amount").alias("minimum_transaction_amount"),

        sum(when(col("transaction_status")=="SUCCESS",1)
            .otherwise(0)).alias("successful_transactions"),

        sum(when(col("transaction_status")=="FAILED",1)
            .otherwise(0)).alias("failed_transactions")
    ))

### 5.2 : Calculate Success Rate

In [0]:
bank_summary = bank_summary.withColumn(
    "success_rate",round((col("successful_transactions")*100)/
                          col("total_transactions"),2))

### 5.3 : Calculate Failure Rate

In [0]:
bank_summary = bank_summary.withColumn(
    "failure_rate",round((col("failed_transactions")*100)/
                          col("total_transactions"),2))

### 5.4 : Verify

In [0]:
bank_summary = bank_summary.orderBy(
    col("total_transactions").desc())

display(bank_summary)

sender_bank,total_transactions,total_amount,average_transaction_amount,maximum_transaction_amount,minimum_transaction_amount,successful_transactions,failed_transactions,success_rate,failure_rate
HDFC,4805,1.22618569E8,25518.95,49985.0,24.0,4651,154,96.8,3.2
ICICI,4800,1.20972497E8,25202.6,49963.0,5.0,4652,148,96.92,3.08
AXIS,4785,1.1954394E8,24983.06,49979.0,9.0,4635,150,96.87,3.13
SBI,4685,1.17325464E8,25042.79,49999.0,10.0,4533,152,96.76,3.24


## Step 6 : City-wise Transaction Summary

### 6.1 : Create City Summary

In [0]:
city_summary = (silver_df.groupBy("city")
    .agg(
        count("*").alias("total_transactions"),

        sum("amount").alias("total_amount"),

        round(avg("amount"),2).alias("average_transaction_amount"),

        max("amount").alias("maximum_transaction_amount"),

        min("amount").alias("minimum_transaction_amount"),

        sum(when(col("transaction_status")=="SUCCESS",1)
            .otherwise(0)).alias("successful_transactions"),

        sum(when(col("transaction_status")=="FAILED",1)
            .otherwise(0)).alias("failed_transactions")
        ))

### 6.2 – Calculate Success Rate

In [0]:
city_summary = city_summary.withColumn(
    "success_rate",round((col("successful_transactions")*100)/
                          col("total_transactions"),2))

### 7.3 : Calculate Failure Rate

In [0]:
city_summary = city_summary.withColumn(
    "failure_rate",round((col("failed_transactions")*100)/
                          col("total_transactions"),2))

### 7.4 : Verify

In [0]:
city_summary = city_summary.orderBy(col("total_transactions").desc())

display(city_summary)

city,total_transactions,total_amount,average_transaction_amount,maximum_transaction_amount,minimum_transaction_amount,successful_transactions,failed_transactions,success_rate,failure_rate
PUNE,2439,6.1377413E7,25164.99,49940.0,9.0,2369,70,97.13,2.87
JAIPUR,2423,6.0426322E7,24938.64,49973.0,32.0,2339,84,96.53,3.47
BANGALORE,2407,5.9614745E7,24767.24,49968.0,23.0,2334,73,96.97,3.03
HYDERABAD,2404,6.0864613E7,25318.06,49951.0,10.0,2333,71,97.05,2.95
INDORE,2387,6.1069281E7,25584.11,49933.0,5.0,2324,63,97.36,2.64
MUMBAI,2363,5.9999688E7,25391.32,49993.0,24.0,2276,87,96.32,3.68
AHMEDABAD,2340,5.9825247E7,25566.34,49999.0,36.0,2249,91,96.11,3.89
DELHI,2312,5.7283161E7,24776.45,49985.0,24.0,2247,65,97.19,2.81


## Step 8 : Transaction Type Summary

### 8.1 : Create Transaction Type Summary

In [0]:
transaction_type_summary = (silver_df.groupBy("transaction_type")
    .agg(
        count("*").alias("total_transactions"),

        sum("amount").alias("total_amount"),

        round(avg("amount"),2).alias("average_transaction_amount"),

        max("amount").alias("maximum_transaction_amount"),

        min("amount").alias("minimum_transaction_amount"),

        sum(when(col("transaction_status")=="SUCCESS",1)
            .otherwise(0)).alias("successful_transactions"),

        sum(when(col("transaction_status")=="FAILED",1)
            .otherwise(0)).alias("failed_transactions")
    ))

### 8.2 : Success Rate

In [0]:
transaction_type_summary = transaction_type_summary.withColumn(
    "success_rate",round((col("successful_transactions")*100)/
                          col("total_transactions"),2))

### 8.3 : Failure Rate

In [0]:
transaction_type_summary = transaction_type_summary.withColumn(
    "failure_rate",round((col("failed_transactions")*100)/
                          col("total_transactions"),2))

### 8.5 – Verify

In [0]:
transaction_type_summary = transaction_type_summary.orderBy(col("total_transactions").desc())

display(transaction_type_summary)

transaction_type,total_transactions,total_amount,average_transaction_amount,maximum_transaction_amount,minimum_transaction_amount,successful_transactions,failed_transactions,success_rate,failure_rate
P2P,9572,2.41043513E8,25182.15,49999.0,24.0,9265,307,96.79,3.21
MERCHANT,3799,9.5960235E7,25259.34,49968.0,5.0,3697,102,97.32,2.68
RECHARGE,2905,7.2780225E7,25053.43,49981.0,32.0,2812,93,96.8,3.2
BILL PAYMENT,2799,7.0676497E7,25250.62,49961.0,36.0,2697,102,96.36,3.64


## Step 9 : Write Gold Tables

### 9.1 : Define Gold Table Names

In [0]:
gold_daily_table = "gold_daily_summary"
gold_bank_table = "gold_bank_summary"
gold_city_table = "gold_city_summary"
gold_transaction_table = "gold_transaction_type_summary"

In [0]:
daily_summary.write \
             .format("delta") \
             .mode("overwrite") \
             .saveAsTable(gold_daily_table)

print("Daily Table Created Successfully")

Daily Table Created Successfully


In [0]:
bank_summary.write \
             .format("delta") \
             .mode("overwrite") \
             .saveAsTable(gold_bank_table)

print("Bank Table Created Successfully")

Bank Table Created Successfully


In [0]:
city_summary.write \
             .format("delta") \
             .mode("overwrite") \
             .saveAsTable(gold_city_table)

print("City Table Created Successfully")

City Table Created Successfully


In [0]:
transaction_type_summary.write \
             .format("delta") \
             .mode("overwrite") \
             .saveAsTable(gold_transaction_table)

print("Transaction Table Created Successfully")

Transaction Table Created Successfully


## Step 10 : Verify Gold Tables

In [0]:
display(spark.table(gold_daily_table))

transaction_date,total_transactions,total_amount,successful_transactions,failed_transactions,average_transaction_amount,maximum_transaction_amount,minimum_transaction_amount,success_rate,failure_rate
2026-06-12,163,4101918.0,157,6,25165.14,49848.0,1094.0,96.32,3.68
2026-06-13,622,1.5424446E7,608,14,24798.14,49963.0,153.0,97.75,2.25
2026-06-14,644,1.6887342E7,623,21,26222.58,49708.0,112.0,96.74,3.26
2026-06-15,595,1.4886322E7,577,18,25019.03,49973.0,201.0,96.97,3.03
2026-06-16,685,1.7581704E7,668,17,25666.72,49963.0,151.0,97.52,2.48
2026-06-17,625,1.5846873E7,605,20,25355.0,49900.0,10.0,96.8,3.2
2026-06-18,684,1.7373323E7,654,30,25399.6,49890.0,28.0,95.61,4.39
2026-06-19,637,1.666387E7,618,19,26159.92,49920.0,48.0,97.02,2.98
2026-06-20,637,1.6351033E7,616,21,25668.81,49944.0,32.0,96.7,3.3
2026-06-21,594,1.481295E7,577,17,24937.63,49953.0,171.0,97.14,2.86


In [0]:
display(spark.table(gold_bank_table))

sender_bank,total_transactions,total_amount,average_transaction_amount,maximum_transaction_amount,minimum_transaction_amount,successful_transactions,failed_transactions,success_rate,failure_rate
HDFC,4805,1.22618569E8,25518.95,49985.0,24.0,4651,154,96.8,3.2
ICICI,4800,1.20972497E8,25202.6,49963.0,5.0,4652,148,96.92,3.08
AXIS,4785,1.1954394E8,24983.06,49979.0,9.0,4635,150,96.87,3.13
SBI,4685,1.17325464E8,25042.79,49999.0,10.0,4533,152,96.76,3.24


In [0]:
display(spark.table(gold_city_table))

city,total_transactions,total_amount,average_transaction_amount,maximum_transaction_amount,minimum_transaction_amount,successful_transactions,failed_transactions,success_rate,failure_rate
PUNE,2439,6.1377413E7,25164.99,49940.0,9.0,2369,70,97.13,2.87
JAIPUR,2423,6.0426322E7,24938.64,49973.0,32.0,2339,84,96.53,3.47
BANGALORE,2407,5.9614745E7,24767.24,49968.0,23.0,2334,73,96.97,3.03
HYDERABAD,2404,6.0864613E7,25318.06,49951.0,10.0,2333,71,97.05,2.95
INDORE,2387,6.1069281E7,25584.11,49933.0,5.0,2324,63,97.36,2.64
MUMBAI,2363,5.9999688E7,25391.32,49993.0,24.0,2276,87,96.32,3.68
AHMEDABAD,2340,5.9825247E7,25566.34,49999.0,36.0,2249,91,96.11,3.89
DELHI,2312,5.7283161E7,24776.45,49985.0,24.0,2247,65,97.19,2.81


In [0]:
display(spark.table(gold_transaction_table))

transaction_type,total_transactions,total_amount,average_transaction_amount,maximum_transaction_amount,minimum_transaction_amount,successful_transactions,failed_transactions,success_rate,failure_rate
P2P,9572,2.41043513E8,25182.15,49999.0,24.0,9265,307,96.79,3.21
MERCHANT,3799,9.5960235E7,25259.34,49968.0,5.0,3697,102,97.32,2.68
RECHARGE,2905,7.2780225E7,25053.43,49981.0,32.0,2812,93,96.8,3.2
BILL PAYMENT,2799,7.0676497E7,25250.62,49961.0,36.0,2697,102,96.36,3.64


### 10.1 : Check Record Counts

In [0]:
print("Daily Summary :",spark.table(gold_daily_table).count())
print("Bank Summary :",spark.table(gold_bank_table).count())
print("City Summary :", spark.table(gold_city_table).count())
print("Transaction Type Summary :", spark.table(gold_transaction_table).count())

Daily Summary : 31
Bank Summary : 4
City Summary : 8
Transaction Type Summary : 4
